## Deteccao e Contabilizacao de Embalagens de Alimentos com YOLOv8

### Objetivo

Treinar um modelo de deteccao de objetos capaz de identificar e contabilizar embalagens de alimentos, calculando automaticamente **quantidade, peso estimado e valor monetario** de cada item detectado. O modelo classifica **5 classes granulares** — arroz, feijao, acucar, cafe e macarrao — mapeadas para **3 categorias** na API: arroz, feijao e outros. Quando a IA reconhece um item especifico dentro de "outros", essa informacao e incluida no campo `sub_item`.

### Metodologia

1. Importacao das bibliotecas e configuracao dos hiperparametros
2. Definicao dos caminhos, constantes e tabelas de peso/preco
3. Funcoes de data augmentation com OpenCV
4. Geracao do dataset aumentado a partir das imagens base
5. Divisao do dataset em treino e validacao (80/20)
6. Geracao das labels no formato YOLO
7. Treinamento do modelo YOLOv8n (transfer learning)
8. Avaliacao das metricas (mAP, precision, recall)
9. Inferencia visual em imagens de validacao

### Mapeamento de Classes

| Classe YOLO | ID | Categoria API | sub_item |
|-------------|----|---------------|----------|
| arroz       | 0  | arroz         | arroz    |
| feijao      | 1  | feijao        | feijao   |
| acucar      | 2  | outros        | acucar   |
| cafe        | 3  | outros        | cafe     |
| macarrao    | 4  | outros        | macarrao |

### Ambiente Controlado

As imagens sao capturadas em ambiente controlado: fundo escuro, iluminacao fixa e camera posicionada sobre uma rampa. Os itens deslizam pela rampa e sao detectados, classificados e contados automaticamente pelo modulo `camera-ai`.

### Por que YOLOv8?

Diferente de classificadores tradicionais (como MobileNetV2), o YOLOv8 realiza **deteccao + classificacao em um unico passo**, fornecendo bounding boxes ao redor dos objetos. O modelo nano (YOLOv8n) tem apenas ~6 MB e executa inferencia em ~55 ms por frame na CPU, ideal para aplicacoes em quase tempo real.

## Sobre este notebook

Este notebook e **executavel** — todos os paths apontam para [`src/Entrega2/backend/camera-ai/training/`](../../../src/Entrega2/backend/camera-ai/training/), onde estao o dataset, modelo e scripts. Basta rodar as celulas em ordem.

Alternativamente, o mesmo pipeline pode ser executado via CLI:

```bash
cd src/Entrega2/backend/camera-ai/training
python generate_dataset.py
python split_dataset.py
python generate_labels.py
python train_yolo.py
python evaluate.py
```

A inferencia em tempo real e feita pelo modulo `camera-ai` (fora deste notebook):

```bash
cd src/Entrega2/backend/camera-ai
python main.py
```

## Passo 1 — Importacao das Bibliotecas

Utilizamos:
- **ultralytics**: framework YOLOv8 para deteccao de objetos
- **OpenCV**: data augmentation e processamento de imagens
- **NumPy**: manipulacao de arrays e operacoes numericas
- **matplotlib**: visualizacao dos resultados de inferencia
- **pathlib / os / shutil / tqdm**: manipulacao de arquivos e progresso

In [ ]:
import os
import random
import shutil
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO

print(f"OpenCV:  {cv2.__version__}")
print(f"NumPy:   {np.__version__}")
print("\nBibliotecas importadas com sucesso.")

## Passo 2 — Caminhos e Constantes

Definimos os diretorios do dataset, hiperparametros de treino e as tabelas de **peso e preco** por classe. Os valores fixos evitam ML desnecessario para um dado que e pre-conhecido e permitem ajuste rapido sem alterar a logica de deteccao.

| Produto  | Peso (g) | Preco/kg (R$) | Valor unit. (R$) |
|----------|----------|---------------|------------------|
| arroz    | 1.000    | 5,50          | 5,50             |
| feijao   | 1.000    | 7,50          | 7,50             |
| acucar   | 1.000    | 4,50          | 4,50             |
| cafe     | 500      | 50,00         | 25,00            |
| macarrao | 500      | 8,00          | 4,00             |

O mapeamento `CLASS_TO_CATEGORY` converte as 5 classes YOLO para as 3 categorias da API, incluindo o `sub_item` que identifica o alimento especifico dentro de "outros".

In [ ]:
NOTEBOOK_DIR = Path(".").resolve()
BASE_DIR = (NOTEBOOK_DIR / ".." / ".." / ".." / "src" / "Entrega2" / "backend" / "camera-ai" / "training").resolve()

DATASET_BASE_DIR = BASE_DIR / "dataset_base"
DATASET_DIR      = BASE_DIR / "dataset"
DATA_YAML        = BASE_DIR / "data.yaml"
RUNS_DIR         = BASE_DIR / "runs"

YOLO_CLASSES = ["arroz", "feijao", "acucar", "cafe", "macarrao"]
CLASS_MAP    = {cls: i for i, cls in enumerate(YOLO_CLASSES)}
CATEGORIES   = ["arroz", "feijao", "outros"]

CLASS_TO_CATEGORY = {
    "arroz":    ("arroz",  "arroz"),
    "feijao":   ("feijao", "feijao"),
    "acucar":   ("outros", "acucar"),
    "cafe":     ("outros", "cafe"),
    "macarrao": ("outros", "macarrao"),
}

CATEGORY_WEIGHTS_G = {
    "arroz":    1000.0,
    "feijao":   1000.0,
    "acucar":   1000.0,
    "cafe":      500.0,
    "macarrao":  500.0,
}

CATEGORY_PRICES_BRL_PER_KG = {
    "arroz":    5.50,
    "feijao":   7.50,
    "acucar":   4.50,
    "cafe":    50.00,
    "macarrao": 8.00,
}

IMG_SIZE         = 640
IMAGES_PER_INPUT = 10
SPLIT_RATIO      = 0.2
SEED             = 42
YOLO_BASE_MODEL  = str(BASE_DIR / "yolov8n.pt")
EPOCHS           = 30
BATCH_SIZE       = 8
TRAIN_NAME       = "treino_alimentos"
CONFIDENCE_THRESHOLD = 0.75
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print(f"Classes YOLO:    {YOLO_CLASSES}")
print(f"Categorias API:  {CATEGORIES}")
print(f"IMG_SIZE:        {IMG_SIZE}")
print(f"Augmentacoes:    {IMAGES_PER_INPUT} por imagem")
print(f"Epochs:          {EPOCHS}  |  Batch: {BATCH_SIZE}")
print(f"Threshold conf:  {CONFIDENCE_THRESHOLD}")
print(f"\nBase dir:        {BASE_DIR}")

print("\nDataset base:")
for cls in YOLO_CLASSES:
    cls_dir = DATASET_BASE_DIR / cls
    count = len([p for p in cls_dir.glob("*") if p.suffix.lower() in SUPPORTED_EXTENSIONS]) if cls_dir.exists() else 0
    weight = CATEGORY_WEIGHTS_G[cls]
    price_unit = round((weight / 1000.0) * CATEGORY_PRICES_BRL_PER_KG[cls], 2)
    print(f"  {cls:<10} {count:>3} imgs base  |  {weight:.0f}g  R${price_unit:.2f}/unid")

## Passo 3 — Data Augmentation com OpenCV

Para aumentar a diversidade do dataset e melhorar a robustez do modelo, aplicamos transformacoes aleatorias em cada imagem original:

| Tecnica           | Parametros                      | Probabilidade |
|-------------------|---------------------------------|---------------|
| Rotacao           | +/- 20 graus                    | 70%           |
| Flip              | Horizontal, vertical, ambos     | 50%           |
| Brilho/Contraste  | alpha [0.8, 1.2], beta [-30,30] | 50%           |
| Blur Gaussiano    | kernel 3x3 ou 5x5               | 30%           |
| Ruido Gaussiano   | sigma = 10                      | 30%           |
| Zoom              | escala [0.85, 1.15]             | 50%           |

Cada imagem gera **10 variantes aumentadas + a original redimensionada**, totalizando ~968 imagens a partir de 88 originais distribuidas nas 5 classes.

In [ ]:
def random_rotate(img):
    if random.random() < 0.7:
        angle = random.uniform(-20, 20)
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        return cv2.warpAffine(img, M, (w, h))
    return img

def random_flip(img):
    if random.random() < 0.5:
        return cv2.flip(img, random.choice([-1, 0, 1]))
    return img

def adjust_brightness_contrast(img):
    if random.random() < 0.5:
        alpha = random.uniform(0.8, 1.2)
        beta  = random.randint(-30, 30)
        return np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    return img

def random_blur(img):
    if random.random() < 0.3:
        k = random.choice([3, 5])
        return cv2.GaussianBlur(img, (k, k), 0)
    return img

def random_noise(img):
    if random.random() < 0.3:
        noise = np.random.normal(0, 10, img.shape).astype(np.int16)
        return np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return img

def random_zoom(img):
    if random.random() < 0.5:
        scale = random.uniform(0.85, 1.15)
        h, w  = img.shape[:2]
        nh, nw = int(h * scale), int(w * scale)
        resized = cv2.resize(img, (nw, nh))
        if scale > 1:
            sy = (nh - h) // 2
            sx = (nw - w) // 2
            return resized[sy:sy + h, sx:sx + w]
        pad_y = (h - nh) // 2
        pad_x = (w - nw) // 2
        out = np.zeros_like(img)
        out[pad_y:pad_y + nh, pad_x:pad_x + nw] = resized
        return out
    return img

def augment_image(img):
    img = random_rotate(img)
    img = random_flip(img)
    img = adjust_brightness_contrast(img)
    img = random_blur(img)
    img = random_noise(img)
    img = random_zoom(img)
    return cv2.resize(img, (IMG_SIZE, IMG_SIZE))

print("Funcoes de augmentation definidas.")

## Passo 4 — Gerar Dataset Aumentado

Aplica as augmentacoes em todas as imagens base e salva o resultado em `dataset/images/train/`. Cada imagem original gera a versao redimensionada (`_orig.jpg`) mais 10 variantes aumentadas (`_aug_0.jpg` a `_aug_9.jpg`).

In [ ]:
for category in YOLO_CLASSES:
    out_dir = DATASET_DIR / "images" / "train" / category
    out_dir.mkdir(parents=True, exist_ok=True)

    src = DATASET_BASE_DIR / category
    if not src.exists():
        print(f"[AVISO] {src} nao encontrado, pulando.")
        continue

    images = [p for p in src.iterdir() if p.suffix.lower() in SUPPORTED_EXTENSIONS]
    for img_path in tqdm(images, desc=f"Augmenting {category}"):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img  = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        base = out_dir / img_path.stem
        cv2.imwrite(str(base) + "_orig.jpg", img)
        for i in range(IMAGES_PER_INPUT):
            cv2.imwrite(str(base) + f"_aug_{i}.jpg", augment_image(img))

    total = len(list(out_dir.glob("*.jpg")))
    print(f"  {category}: {len(images)} originais -> {total} imagens geradas")

print("\nDataset gerado.")

## Passo 5 — Dividir em Treino e Validacao

Separamos 20% das imagens para validacao. As imagens sao movidas de `dataset/images/train/` para `dataset/images/val/`, mantendo a estrutura de categorias. O resultado esperado e aproximadamente:

| Split  | Imagens |
|--------|---------|
| Treino | ~774    |
| Val    | ~194    |

In [ ]:
random.seed(SEED)

for category in YOLO_CLASSES:
    category_path     = DATASET_DIR / "images" / "train" / category
    val_category_path = DATASET_DIR / "images" / "val"   / category
    val_category_path.mkdir(parents=True, exist_ok=True)

    if not category_path.exists():
        continue

    imgs = [p for p in category_path.iterdir() if p.suffix.lower() in SUPPORTED_EXTENSIONS]
    random.shuffle(imgs)
    val_imgs = imgs[:int(len(imgs) * SPLIT_RATIO)]

    for p in val_imgs:
        shutil.move(str(p), str(val_category_path / p.name))

    train_count = len(list(category_path.glob("*.jpg")))
    val_count   = len(list(val_category_path.glob("*.jpg")))
    print(f"  {category}: {train_count} treino / {val_count} validacao")

print("\nDivisao train/val concluida.")

## Passo 6 — Gerar Labels YOLO

Para cada imagem, criamos um arquivo `.txt` no formato YOLO:
```
<classe> <x_centro> <y_centro> <largura> <altura>
```
Os valores sao normalizados entre 0 e 1. Como os itens sao centralizados na rampa (ambiente controlado), usamos bounding box fixa `0.5 0.5 0.8 0.8` — o objeto ocupa ~80% do frame centralizado.

Tambem geramos o arquivo `data.yaml` que o YOLOv8 usa para localizar o dataset e mapear os IDs de classe para os nomes.

In [ ]:
for split in ["train", "val"]:
    for category in YOLO_CLASSES:
        img_dir = DATASET_DIR / "images" / split / category
        lbl_dir = DATASET_DIR / "labels" / split / category
        lbl_dir.mkdir(parents=True, exist_ok=True)

        if not img_dir.exists():
            continue

        class_id = CLASS_MAP[category]
        for img_path in img_dir.iterdir():
            if img_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
                continue
            (lbl_dir / (img_path.stem + ".txt")).write_text(f"{class_id} 0.5 0.5 0.8 0.8\n")

DATA_YAML.write_text(
    f"path: {DATASET_DIR.resolve()}\ntrain: images/train\nval: images/val\n\nnames:\n"
    + "".join(f"  {i}: {cls}\n" for i, cls in enumerate(YOLO_CLASSES))
)

print("Labels e data.yaml gerados.")
print(f"\nConteudo do data.yaml:")
print(DATA_YAML.read_text())

## Passo 7 — Treinamento do YOLOv8

Utilizamos o **YOLOv8n** (nano) pre-treinado no COCO dataset como base. O modelo e retreinado com nosso dataset de alimentos por 30 epocas. O YOLOv8 gerencia automaticamente:
- Redimensionamento e normalizacao das imagens
- Augmentacoes internas adicionais (mosaic, mixup)
- Salvamento do melhor modelo por mAP (`best.pt`)
- Logs de treinamento e curvas de aprendizado em `runs/detect/treino_alimentos/`

**Hiperparametros:**
- Otimizador: AdamW (selecionado automaticamente)
- Dispositivo: CPU | AMP (mixed precision) ativado
- Tempo estimado: ~50 min em CPU

In [ ]:
model = YOLO(YOLO_BASE_MODEL)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    name=TRAIN_NAME,
    project=str(RUNS_DIR / "detect"),
)

print("\nTreinamento concluido.")
print(f"Modelo salvo em: runs/detect/{TRAIN_NAME}/weights/best.pt")

## Passo 8 — Avaliacao do Modelo

Carregamos o melhor modelo treinado (`best.pt`) e executamos a validacao sobre o conjunto de validacao para obter as metricas:

- **mAP50**: mean Average Precision com IoU >= 0.50 (metrica principal de deteccao)
- **mAP50-95**: mean Average Precision com IoU de 0.50 a 0.95 (mais rigoroso)
- **Precision**: dos itens detectados, quantos eram corretos
- **Recall**: dos itens existentes, quantos foram detectados

Resultados esperados (extraidos do `best.pt` do treino realizado):

| Metrica  | Valor |
|----------|-------|
| mAP50    | 99,5% |
| mAP50-95 | 99,5% |
| Precision| 99,7% |
| Recall   | 99,9% |

In [ ]:
weights_path = RUNS_DIR / "detect" / TRAIN_NAME / "weights" / "best.pt"
eval_model   = YOLO(str(weights_path))
metrics      = eval_model.val(data=str(DATA_YAML), imgsz=IMG_SIZE)

print("\n=== Metricas do Modelo ===")
print(f"mAP50:    {metrics.box.map50:.4f}  ({metrics.box.map50 * 100:.1f}%)")
print(f"mAP50-95: {metrics.box.map:.4f}  ({metrics.box.map * 100:.1f}%)")
print()

class_names = eval_model.names
print(f"{'Classe':<12} {'Precision':>10} {'Recall':>10} {'AP50':>10}")
print("-" * 46)
for i, (p, r, ap50) in enumerate(zip(metrics.box.p, metrics.box.r, metrics.box.ap50)):
    name = class_names.get(i, f"classe_{i}")
    print(f"{name:<12} {p:>10.4f} {r:>10.4f} {ap50:>10.4f}")

## Passo 9 — Inferencia Visual em Imagens de Validacao

Rodamos o modelo sobre amostras do conjunto de validacao e exibimos:
- A imagem anotada com a bounding box e o label detectado
- A classe detectada (raw_label), a categoria da API e o sub_item
- O **peso estimado** e o **valor calculado** para aquela deteccao

Isso demonstra o pipeline completo: da imagem crua ate a contabilizacao em R$.

In [ ]:
sample_images = []
for category in YOLO_CLASSES:
    imgs = list((DATASET_DIR / "images" / "val" / category).glob("*.jpg"))
    if imgs:
        sample_images.append((category, imgs[0]))

fig, axes = plt.subplots(1, len(sample_images), figsize=(5 * len(sample_images), 5))
if len(sample_images) == 1:
    axes = [axes]

for ax, (category, img_path) in zip(axes, sample_images):
    res      = eval_model(str(img_path), conf=CONFIDENCE_THRESHOLD, verbose=False)
    annotated = res[0].plot()

    cat_api, sub_item = CLASS_TO_CATEGORY.get(category, ("outros", "desconhecido"))
    weight_g  = CATEGORY_WEIGHTS_G.get(category, 0)
    price_brl = round((weight_g / 1000.0) * CATEGORY_PRICES_BRL_PER_KG.get(category, 0), 2)

    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(
        f"{category}  ({cat_api})\n"
        f"{weight_g:.0f}g  |  R${price_brl:.2f}",
        fontsize=10,
    )
    ax.axis("off")

plt.suptitle("Inferencia visual — conjunto de validacao", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\nResumo de pesos e valores por classe:")
print(f"{'Classe':<12} {'Peso':>8} {'Preco/kg':>10} {'Valor unit':>12}")
print("-" * 46)
for cls in YOLO_CLASSES:
    w = CATEGORY_WEIGHTS_G[cls]
    p = CATEGORY_PRICES_BRL_PER_KG[cls]
    v = round((w / 1000.0) * p, 2)
    print(f"{cls:<12} {w:>6.0f}g  {p:>8.2f}/kg  R${v:>8.2f}")

## Conclusao

Neste notebook implementamos o pipeline completo de treinamento para deteccao e contabilizacao de embalagens de alimentos:

1. **Data Augmentation**: 6 tecnicas via OpenCV (rotacao, flip, brilho/contraste, blur, ruido, zoom) expandiram 88 imagens base para ~968
2. **YOLOv8n**: modelo nano pre-treinado, retreinado com 5 classes granulares em 30 epocas — mAP50 de 99,5%
3. **Mapeamento inteligente**: 5 classes YOLO -> 3 categorias de API com campo `sub_item` para granularidade
4. **Contabilizacao automatica**: peso e valor unitario fixos por classe, calculados em write-time sem ML adicional

### Pipeline de inferencia em tempo real (camera-ai)

O modelo `best.pt` gerado aqui e usado pelo modulo `camera-ai` para:

- **Deteccao** (`ml/inference.py`): YOLOv8 analisa cada frame com confianca >= 0,75
- **Tracking** (`tracking/tracker.py`): CentroidTracker associa deteccoes entre frames por distancia euclidiana (limite 100 px, descarta apos 5 frames)
- **Estabilidade**: track precisa manter a mesma classe por 10 frames consecutivos
- **Linha virtual** (`tracking/line_counter.py`): item e contabilizado ao cruzar a linha horizontal (50% da altura), usando o ID do track para prevenir dupla contagem
- **Persistencia** (`services/detection_writer.py`): cada contagem confirmada e gravada na tabela `ai_detections` do PostgreSQL com timestamp, peso, valor e equipe
- **Evidencia** (`services/s3_uploader.py`): frame anotado enviado em background ao Amazon S3

Esse ciclo garante que cada pacote seja contado uma unica vez e que cada contagem fique registrada com sua respectiva evidencia visual.